In [13]:
# ============================================================================
# source_Identification_Sorting_And_Photometry_Algorithm.ipynb - ALL-IN-ONE CELL
# ============================================================================

from pathlib import Path
from datetime import datetime
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits as astrofits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.stats import sigma_clipped_stats, SigmaClip
from photutils.aperture import (CircularAperture, CircularAnnulus,
                                 ApertureStats, aperture_photometry)
from photutils.centroids import centroid_sources, centroid_com
from photutils.background import Background2D, MedianBackground
import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================================================
# PATHS
# ============================================================================
UNIT_TEST_DIR = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\unit_Tests")
CUTOUTS_DIR = UNIT_TEST_DIR / 'cutouts'
CATALOG_PATH = UNIT_TEST_DIR / 'synthetic_catalog.csv'

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = UNIT_TEST_DIR / f'test_Algorithm_{RUN_TIMESTAMP}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = UNIT_TEST_DIR / 'algorithm_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / 'algorithm_photometry_results.csv'
LIGHTCURVE_PATH = OUTPUT_DIR / 'algorithm_lightcurve.csv'
REF_PHOTOMETRY_PATH = OUTPUT_DIR / 'algorithm_reference_photometry.csv'

print(f"This run's output folder: {OUTPUT_DIR}")

# ============================================================================
# PHOTOMETRY PARAMETERS
# ============================================================================
DEFAULT_APERTURE = 6
ANNULUS_INNER = 15
ANNULUS_OUTER = 20
CENTROID_BOX = 21
CENTROID_MAX_DRIFT = 5
SIGNIF_THRESHOLD = 5.0
SATURATION_FRACTION = 0.99
SATURATION_MIN_PIXELS = 5

ISOLATION_MIN_SEPARATION = ANNULUS_OUTER + 2

MIN_REF_STARS = 10
OUTLIER_SIGMA = 3.0
MIN_ABS_SLOPE_FOR_INVERSION = 0.3

REF_STARS_MAX_PER_PLATE = 150

VERSION = 12  # ref_catalog excludes phantom stars (root cause fix; median
              # slope ~0.98 confirmed across ~600 verified plates)

# ============================================================================
# LOAD STATIC FIELD CATALOG
# ============================================================================
catalog = pd.read_csv(CATALOG_PATH)
target_row = catalog[catalog['is_target'] == True].iloc[0]
TARGET_COORD = SkyCoord(ra=target_row['ra'] * u.deg, dec=target_row['dec'] * u.deg)

ref_catalog = catalog[(catalog['is_target'] == False) & (catalog['is_phantom'] == False)].copy()
print(f"Loaded catalog: {len(catalog)} stars ({len(ref_catalog)} non-phantom reference candidates, "
      f"{(catalog['is_target'] == False).sum() - len(ref_catalog)} phantoms excluded)")
print(f"Target coordinate: RA={TARGET_COORD.ra.deg:.6f}, Dec={TARGET_COORD.dec.deg:.6f}")

cutouts_all = sorted(CUTOUTS_DIR.glob("*.fits"))
print(f"Found {len(cutouts_all)} total plates available")

# --- SUBSET MODE (kept for fast iteration; set to None for the full run) ---
SUBSET_SIZE = None  # e.g. 800 for a quick subset test; None = full dataset
if SUBSET_SIZE is not None:
    rng_subset = np.random.default_rng(42)  # fixed seed so the subset is stable across reruns
    cutouts = list(rng_subset.choice(cutouts_all, size=min(SUBSET_SIZE, len(cutouts_all)), replace=False))
    cutouts = sorted(cutouts, key=lambda p: p.name)
    print(f"Using a random subset of {len(cutouts)} plates for this run")
else:
    cutouts = cutouts_all
    print(f"Using all {len(cutouts)} plates")

def load_cache(name, default=None):
    path = CACHE_DIR / f"{name}.pkl"
    if path.exists():
        try:
            with open(path, 'rb') as f:
                return pickle.load(f)
        except Exception:
            pass
    return default if default is not None else {}

def save_cache(name, data):
    with open(CACHE_DIR / f"{name}.pkl", 'wb') as f:
        pickle.dump(data, f)

plate_db = load_cache('plate_db', {})
print(f"Loaded {len(plate_db)} cached plate results")


# ============================================================================
# CORE PHOTOMETRY & CALIBRATION FUNCTIONS
# ============================================================================

def subtract_background_2d(data, box_size=50):
    try:
        bkg = Background2D(data, (box_size, box_size), filter_size=(3, 3),
                            bkg_estimator=MedianBackground())
        return data - bkg.background, bkg.background_rms
    except Exception:
        med = np.nanmedian(data)
        return data - med, np.full_like(data, np.nanstd(data))

def refine_centroids(data, x, y, box_size=CENTROID_BOX, max_drift=CENTROID_MAX_DRIFT):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) == 0:
        return x, y
    try:
        x_ref, y_ref = centroid_sources(data, x, y, box_size=box_size,
                                         centroid_func=centroid_com)
        good = np.isfinite(x_ref) & np.isfinite(y_ref)
        drift = np.hypot(x_ref - x, y_ref - y)
        good &= drift < max_drift
        x_ref[~good] = x[~good]
        y_ref[~good] = y[~good]
        return x_ref, y_ref
    except Exception:
        return x, y

def filter_isolated_stars(x, y, min_separation=ISOLATION_MIN_SEPARATION):
    n = len(x)
    if n < 2:
        return np.ones(n, dtype=bool)
    isolated = np.ones(n, dtype=bool)
    for i in range(n):
        d = np.hypot(x - x[i], y - y[i])
        d[i] = np.inf
        if np.any(d < min_separation):
            isolated[i] = False
    return isolated

def measure_aperture_photometry(data, x, y, radius=DEFAULT_APERTURE,
                                 annulus_inner=ANNULUS_INNER, annulus_outer=ANNULUS_OUTER):
    n = len(x)
    if n == 0:
        return np.array([]), np.array([]), np.array([])
    
    positions = list(zip(x, y))
    aperture = CircularAperture(positions, r=radius)
    annulus = CircularAnnulus(positions, r_in=annulus_inner, r_out=annulus_outer)
    
    ann_stats = ApertureStats(data, annulus, sigma_clip=SigmaClip(sigma=3.0))
    bkg_median = np.nan_to_num(ann_stats.median, nan=0.0)
    bkg_std = np.nan_to_num(ann_stats.std, nan=0.0)
    
    phot = aperture_photometry(data, aperture)
    flux = phot['aperture_sum'].value - bkg_median * (np.pi * radius**2)
    
    return flux, bkg_median, bkg_std

def detect_saturation(data, x, y, radius=DEFAULT_APERTURE,
                       sat_fraction=SATURATION_FRACTION, min_pixels=SATURATION_MIN_PIXELS):
    n = len(x)
    is_sat = np.zeros(n, dtype=bool)
    data_max = np.nanmax(data)
    if data_max <= 0:
        return is_sat
    
    positions = list(zip(x, y))
    aperture = CircularAperture(positions, r=radius)
    aperture_masks = aperture.to_mask(method='center')
    for i in range(n):
        mask = aperture_masks[i]
        cut = mask.multiply(data)
        if cut is None:
            continue
        star_pixels = cut[mask.data > 0]
        if len(star_pixels) == 0:
            continue
        near_max = np.sum(star_pixels >= sat_fraction * data_max)
        is_sat[i] = near_max >= min_pixels
    return is_sat

def calibrate_photometry(inst_mags, apass_b, outlier_sigma=OUTLIER_SIGMA, min_stars=MIN_REF_STARS):
    mask = np.isfinite(inst_mags) & np.isfinite(apass_b)
    if mask.sum() < min_stars:
        return None
    
    x = apass_b[mask]
    y = inst_mags[mask]
    
    for _ in range(3):
        try:
            coeffs = np.polyfit(x, y, 1)
            residuals = y - (coeffs[0] * x + coeffs[1])
            rms = np.sqrt(np.mean(residuals**2))
            good = np.abs(residuals) < outlier_sigma * rms
            if good.sum() < min_stars:
                break
            x, y = x[good], y[good]
        except Exception:
            return None
    
    if len(x) < min_stars:
        return None
    
    coeffs = np.polyfit(x, y, 1)
    residuals = y - (coeffs[0] * x + coeffs[1])
    rms = np.sqrt(np.mean(residuals**2))
    
    return {'slope': coeffs[0], 'intercept': coeffs[1], 'rms': rms, 'n_used': len(x)}


# ============================================================================
# PER-PLATE PROCESSING PIPELINE
# ============================================================================

def process_plate(fits_path):
    try:
        data = astrofits.getdata(fits_path).astype(float)
        header = astrofits.getheader(fits_path)
        wcs = WCS(header)
        
        jd = header.get('JD-OBS', np.nan)
        
        data_sub, bkg_rms = subtract_background_2d(data)
        
        ra = ref_catalog['ra'].values
        dec = ref_catalog['dec'].values
        star_ids = ref_catalog['id'].values
        apass_b = ref_catalog['apass_b_mag'].values
        
        x, y = wcs.all_world2pix(ra, dec, 0)
        margin = 20
        in_frame = (x > margin) & (x < data.shape[1] - margin) & \
                   (y > margin) & (y < data.shape[0] - margin)
        x, y = x[in_frame], y[in_frame]
        star_ids = star_ids[in_frame]
        apass_b = apass_b[in_frame]
        
        if len(x) > REF_STARS_MAX_PER_PLATE:
            keep = np.random.choice(len(x), size=REF_STARS_MAX_PER_PLATE, replace=False)
            x, y = x[keep], y[keep]
            star_ids = star_ids[keep]
            apass_b = apass_b[keep]
        
        if len(x) == 0:
            return _empty_result(fits_path, jd, 'No reference stars in frame')
        
        x_ref, y_ref = refine_centroids(data_sub, x, y)
        isolated = filter_isolated_stars(x_ref, y_ref)
        flux, bkg_med, bkg_std = measure_aperture_photometry(data_sub, x_ref, y_ref)
        
        area = np.pi * DEFAULT_APERTURE**2
        with np.errstate(divide='ignore', invalid='ignore'):
            snr = np.where(bkg_std > 0, flux / (bkg_std * np.sqrt(area)), 0)
        
        is_sat = detect_saturation(data, x_ref, y_ref)
        
        with np.errstate(divide='ignore', invalid='ignore'):
            inst_mag = np.where(flux > 0, -2.5 * np.log10(np.maximum(flux, 1e-10)), np.nan)
        
        good_for_fit = (snr >= SIGNIF_THRESHOLD) & np.isfinite(inst_mag) & ~is_sat & isolated
        
        calibration = calibrate_photometry(inst_mag[good_for_fit], apass_b[good_for_fit])
        n_ref_used = calibration['n_used'] if calibration else int(good_for_fit.sum())
        
        ref_rows = []
        for i in range(len(x_ref)):
            if np.isfinite(inst_mag[i]) and snr[i] >= SIGNIF_THRESHOLD and isolated[i]:
                ref_rows.append({
                    'filename': fits_path.name,
                    'star_id': star_ids[i],
                    'instrumental_mag': inst_mag[i]
                })
        
        tx, ty = wcs.all_world2pix(TARGET_COORD.ra.deg, TARGET_COORD.dec.deg, 0)
        target_detected = False
        target_mag = np.nan
        target_mag_error = np.nan
        
        if margin < tx < data.shape[1] - margin and margin < ty < data.shape[0] - margin:
            tx_ref, ty_ref = refine_centroids(data_sub, np.array([tx]), np.array([ty]))
            t_flux, t_bkg_med, t_bkg_std = measure_aperture_photometry(data_sub, tx_ref, ty_ref)
            t_area = np.pi * DEFAULT_APERTURE**2
            t_snr = t_flux[0] / (t_bkg_std[0] * np.sqrt(t_area)) if t_bkg_std[0] > 0 else 0
            
            if (t_snr >= SIGNIF_THRESHOLD and t_flux[0] > 0 and calibration is not None
                    and abs(calibration['slope']) >= MIN_ABS_SLOPE_FOR_INVERSION):
                t_inst_mag = -2.5 * np.log10(t_flux[0])
                target_mag = (t_inst_mag - calibration['intercept']) / calibration['slope']
                target_mag_error = np.sqrt((1.0857 / max(t_snr, 1e-6))**2 + calibration['rms']**2)
                target_detected = True
                tx_out, ty_out = float(tx_ref[0]), float(ty_ref[0])
            else:
                tx_out, ty_out = float(tx_ref[0]), float(ty_ref[0])
        else:
            tx_out, ty_out = np.nan, np.nan
        
        quality_ok = (
            calibration is not None and
            n_ref_used >= MIN_REF_STARS and
            target_detected
        )
        
        result_row = {
            'filename': fits_path.name,
            'jd': jd,
            'target_detected': target_detected,
            'target_mag': target_mag,
            'target_mag_error': target_mag_error,
            'target_x': tx_out,
            'target_y': ty_out,
            'num_reference_stars': n_ref_used,
            'zeropoint': calibration['intercept'] if calibration else np.nan,
            'rms_scatter': calibration['rms'] if calibration else np.nan,
            'slope': calibration['slope'] if calibration else np.nan,
            'quality_ok': quality_ok,
        }
        
        return {'result_row': result_row, 'ref_rows': ref_rows, 'calibration': calibration}
        
    except Exception as e:
        print(f"Error processing {fits_path.name}: {e}")
        return _empty_result(fits_path, np.nan, str(e))

def _empty_result(fits_path, jd, reason):
    return {
        'result_row': {
            'filename': fits_path.name, 'jd': jd, 'target_detected': False,
            'target_mag': np.nan, 'target_mag_error': np.nan,
            'target_x': np.nan, 'target_y': np.nan,
            'num_reference_stars': 0, 'zeropoint': np.nan,
            'rms_scatter': np.nan, 'slope': np.nan, 'quality_ok': False,
        },
        'ref_rows': [],
        'calibration': None,
    }


# ============================================================================
# RUN PIPELINE WITH PROGRESS BAR & SAVE OUTPUTS
# ============================================================================

progress_bar = widgets.IntProgress(value=0, min=0, max=len(cutouts), description='Processing:')
progress_html = widgets.HTML(value="")
progress_status = widgets.Label(value="Not started")
display(widgets.VBox([progress_bar, progress_html, progress_status]))

def format_eta(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return "calculating..."
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"

def render_progress(n_done, total, start_time):
    elapsed = time.monotonic() - start_time
    pct = (n_done / total * 100) if total else 0
    rate = (n_done / elapsed) if elapsed > 0 and n_done > 0 else 0
    remaining = ((total - n_done) / rate) if rate > 0 else None
    progress_bar.value = n_done
    progress_html.value = (
        f"<div style='font-family: monospace; font-size: 13px;'>"
        f"<b>{pct:5.1f}%</b> &nbsp; {n_done:,} / {total:,} plates &nbsp;|&nbsp; "
        f"{rate:.2f} plates/sec &nbsp;|&nbsp; "
        f"Elapsed {format_eta(elapsed)} &nbsp;|&nbsp; ETA {format_eta(remaining)}"
        f"</div>"
    )

def run_pipeline(force_reprocess=False):
    global plate_db
    
    total = max(len(cutouts), 1)
    progress_bar.max = total
    progress_bar.value = 0
    start = time.monotonic()
    last_render = 0.0
    n_errors = 0
    
    for i, f in enumerate(cutouts):
        cached = plate_db.get(str(f))
        if not force_reprocess and cached and cached.get('_version', 0) == VERSION:
            pass
        else:
            progress_status.value = f"Processing: {f.name}"
            out = process_plate(f)
            if out['result_row'].get('num_reference_stars', 0) == 0 and out['calibration'] is None:
                n_errors += 1
            out['_version'] = VERSION
            plate_db[str(f)] = out
        
        now = time.monotonic()
        if now - last_render > 0.15 or (i + 1) == total:
            render_progress(i + 1, total, start)
            last_render = now
        
        if (i + 1) % 500 == 0:
            save_cache('plate_db', plate_db)
    
    save_cache('plate_db', plate_db)
    progress_status.value = f"Done. {len(plate_db)} plates in cache ({n_errors} flagged with no usable reference stars)."
    print(progress_status.value)
    
    return _save_outputs()

def _save_outputs():
    result_rows = []
    ref_rows_all = []
    
    for f_str, out in plate_db.items():
        if out is None:
            continue
        result_rows.append(out['result_row'])
        ref_rows_all.extend(out['ref_rows'])
    
    results_df = pd.DataFrame(result_rows).sort_values('jd').reset_index(drop=True)
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Saved {RESULTS_PATH.name} ({len(results_df)} plates)")
    
    ref_df = pd.DataFrame(ref_rows_all)
    ref_df.to_csv(REF_PHOTOMETRY_PATH, index=False)
    print(f"Saved {REF_PHOTOMETRY_PATH.name} ({len(ref_df)} reference measurements)")
    
    lc_df = results_df[results_df['quality_ok']].copy()
    lc_df = lc_df.rename(columns={'target_mag': 'magnitude', 'target_mag_error': 'magnitude_error'})
    lc_df = lc_df[['jd', 'magnitude', 'magnitude_error']].dropna()
    lc_df.to_csv(LIGHTCURVE_PATH, index=False)
    print(f"Saved {LIGHTCURVE_PATH.name} ({len(lc_df)} light curve points)")
    
    return results_df, ref_df, lc_df

results_df, ref_df, lc_df = run_pipeline(force_reprocess=False)


# ============================================================================
# AUTOMATIC SELF-CHECK (folds in the manual diagnostics from earlier —
# no separate cells needed; only reports on plates actually run this session)
# ============================================================================
print("\n" + "=" * 70)
print("SELF-CHECK: calibration quality across plates in this run")
print("=" * 70)

cutout_keys_this_run = set(str(f) for f in cutouts)
cal_rows = []
for f_str, out in plate_db.items():
    if f_str not in cutout_keys_this_run:
        continue
    if out is None or out.get('calibration') is None:
        continue
    cal_rows.append({'slope': out['calibration']['slope'], 'rms': out['calibration']['rms']})
cal_df = pd.DataFrame(cal_rows)

if len(cal_df) > 0:
    print(f"{len(cal_df)} plates (of {len(cutouts)} in this run) produced a calibration fit")
    print(cal_df.describe(percentiles=[.1, .5, .9]))
    median_slope = cal_df['slope'].median()
    if median_slope < 0.7:
        print(f"\n⚠ WARNING: median slope ({median_slope:.3f}) is far from the ideal (1.0).")
        print("  This previously indicated phantom stars contaminating ref_catalog —")
        print("  re-check that ref_catalog filters is_phantom == False before investigating further.")
    else:
        print(f"\n✓ Median slope {median_slope:.3f} — within expected range.")
else:
    print("⚠ No plates in this run produced a calibration fit — check MIN_REF_STARS and SNR thresholds.")

print(f"\nLight curve: {len(lc_df)} points ready for validation")
if SUBSET_SIZE is not None:
    print(f"NOTE: this was a SUBSET run ({len(cutouts)} of {len(cutouts_all)} plates).")
    print("      Cell 2 of unit_Test_Generator.ipynb will still work, but pass rate,")
    print("      completeness, and the confusion matrix will look artificially low —")
    print("      only trust the linearity check and per-error-type breakdown on a subset.")
    print("      Set SUBSET_SIZE = None above and rerun for a full, trustworthy validation.")
else:
    print("Results will be picked up automatically by unit_Test_Generator.ipynb Cell 2")
    print("(set USE_SIMULATED_INPUTS = False there, no other changes needed)")


# ============================================================================
# INTERACTIVE VIEWER: FULL LIGHT CURVE + PER-PLATE LINEARITY CHECK
# ============================================================================

plate_dropdown = widgets.Dropdown(
    options=[(f.name, f) for f in cutouts],
    description="Plate:"
)
toggle_btn = widgets.ToggleButton(
    value=False,
    description="Show Light Curve & Linearity Check",
    button_style='info',
    icon='eye'
)
plot_output = widgets.Output()

def _plot_light_curve_and_linearity(fits_path):
    out = plate_db.get(str(fits_path))
    if out is None:
        print("No cached result for this plate — run the pipeline first.")
        return
    
    row = out['result_row']
    calibration = out['calibration']
    this_jd = row['jd']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    lc_full = results_df[results_df['quality_ok']].sort_values('jd')
    
    if len(lc_full) > 0:
        ax1.errorbar(lc_full['jd'], lc_full['target_mag'],
                     yerr=lc_full['target_mag_error'],
                     fmt='o', color='#3498db', markersize=4, capsize=2, alpha=0.6,
                     label='Light curve')
    else:
        ax1.text(0.5, 0.5, 'No quality-passing plates yet', ha='center', va='center',
                 transform=ax1.transAxes)
    
    if row['quality_ok'] and np.isfinite(row['target_mag']):
        ax1.scatter([this_jd], [row['target_mag']], s=120, facecolors='none',
                    edgecolors='red', linewidths=2, zorder=5, label='This plate')
    else:
        ax1.axvline(this_jd, color='red', linestyle='--', alpha=0.5, label='This plate (excluded)')
    
    ax1.invert_yaxis()
    ax1.set_xlabel('JD')
    ax1.set_ylabel('Target Magnitude')
    ax1.set_title(f'Full Light Curve ({len(lc_full)} points)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    plate_refs = ref_df[ref_df['filename'] == Path(fits_path).name]
    if len(plate_refs) > 0:
        merged = plate_refs.merge(
            ref_catalog[['id', 'apass_b_mag']].rename(columns={'id': 'star_id'}),
            on='star_id', how='left'
        )
        ax2.scatter(merged['apass_b_mag'], merged['instrumental_mag'],
                   alpha=0.5, s=20, color='#2ecc71')
        if calibration is not None:
            x_fit = np.linspace(merged['apass_b_mag'].min(), merged['apass_b_mag'].max(), 100)
            y_fit = calibration['slope'] * x_fit + calibration['intercept']
            ax2.plot(x_fit, y_fit, 'k--', linewidth=2,
                     label=f"slope={calibration['slope']:.3f}, RMS={calibration['rms']:.3f}")
            ax2.legend()
    else:
        ax2.text(0.5, 0.5, 'No reference photometry for this plate', ha='center', va='center',
                 transform=ax2.transAxes)
    ax2.set_xlabel('APASS B Magnitude')
    ax2.set_ylabel('Instrumental Magnitude')
    ax2.set_title(f'Linearity Check ({row["num_reference_stars"]} refs used, isolated only)')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def on_toggle(change):
    with plot_output:
        clear_output(wait=True)
        if toggle_btn.value:
            toggle_btn.description = "Hide Light Curve & Linearity Check"
            toggle_btn.icon = 'eye-slash'
            _plot_light_curve_and_linearity(plate_dropdown.value)
        else:
            toggle_btn.description = "Show Light Curve & Linearity Check"
            toggle_btn.icon = 'eye'

def on_plate_change(change):
    if toggle_btn.value:
        on_toggle(None)

toggle_btn.observe(on_toggle, names='value')
plate_dropdown.observe(on_plate_change, names='value')

display(widgets.VBox([
    widgets.HTML("<b>Per-Plate Light Curve & Linearity Check</b>"),
    widgets.HBox([plate_dropdown, toggle_btn]),
    plot_output
]))

This run's output folder: C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\unit_Tests\test_Algorithm_20260730_030651
Loaded catalog: 801 stars (704 non-phantom reference candidates, 96 phantoms excluded)
Target coordinate: RA=176.349310, Dec=1.544350
Found 5000 total plates available
Using all 5000 plates
Loaded 600 cached plate results


Done. 5000 plates in cache (0 flagged with no usable reference stars).
Saved algorithm_photometry_results.csv (5000 plates)
Saved algorithm_reference_photometry.csv (167288 reference measurements)
Saved algorithm_lightcurve.csv (4064 light curve points)

SELF-CHECK: calibration quality across plates in this run
4990 plates (of 5000 in this run) produced a calibration fit
             slope          rms
count  4990.000000  4990.000000
mean      0.881586     0.486404
std       0.212635     0.477216
min      -0.578331     0.022345
10%       0.546547     0.097503
50%       0.986531     0.255237
90%       1.018145     1.303633
max       1.062510     2.581493

✓ Median slope 0.987 — within expected range.

Light curve: 4064 points ready for validation
Results will be picked up automatically by unit_Test_Generator.ipynb Cell 2
(set USE_SIMULATED_INPUTS = False there, no other changes needed)
